# AMMS 302 — Week 8: Finding Insights with SQL
**Subquery · NOT IN trap · Correlated · CASE WHEN · CTE (WITH) · ROW_NUMBER/RANK · AVG OVER() · Cohort · Time Trends**

> ต่อจาก `healthinfo.db` (สัปดาห์ 5–7) — รันออฟไลน์ด้วย `uv run jupyter lab` · เปิดคู่กับ [สไลด์ wk08](./wk08.html)
> บทนี้รัน query เดียวกันได้บน **SQLite · DuckDB · PostgreSQL** (ดู §11)

### 🎯 Learning objectives (CLO1/CLO2)
- เปลี่ยนคำถามเชิงคลินิก → query ได้ (เช่น "หญิงเบาหวานควบคุมไม่ได้กี่คน?")
- เขียน `IN / NOT EXISTS / correlated`, `CASE WHEN`, `WITH (CTE)`, `ROW_NUMBER / RANK / AVG OVER()` ได้
- สืบค้นเชิงลึกบน `healthinfo` (patients ⨯ prescriptions ⨯ diagnoses) พร้อม sanity-check จำนวนแถวทุกครั้ง
- รัน query เดียวกันบน 3 เอนจิน (SQLite/DuckDB/PostgreSQL) และอธิบายความต่างได้

### 🗺️ แผนที่สไลด์ ↔ โน้ตบุ๊ก
| ipynb § | หัวข้อ | สไลด์ wk08 |
|---|---|---|
| §1 | Why Insights | 02 |
| §2 | Subquery (IN / scalar / table) | 03 |
| §3 | NOT IN trap vs EXISTS / anti-join | 04 |
| §4 | Correlated vs JOIN | 05 |
| §5 | CASE WHEN (bucket) | 06 |
| §6 | CTE (WITH) | 07 |
| §7 | ROW_NUMBER / RANK / DENSE_RANK | 08 |
| §8 | Window aggregate AVG OVER() | 09 |
| §9 | Cohort รวมเครื่องมือ | 10 |
| §10 | Time trends | 11 |
| §11 | 3 เอนจิน: SQLite · DuckDB · PostgreSQL | 12 |

### 📚 Official references
- SQLite: [SELECT](https://www.sqlite.org/lang_select.html) · [Subquery](https://www.sqlite.org/lang_expr.html#subquery) · [EXISTS](https://www.sqlite.org/lang_expr.html#exists) · [CASE](https://www.sqlite.org/lang_expr.html#case) · [WITH](https://www.sqlite.org/lang_with.html) · [Window Functions](https://www.sqlite.org/windowfunctions.html) · [Date/Time](https://www.sqlite.org/lang_datefunc.html)
- DuckDB: [SQL introduction](https://duckdb.org/docs/stable/sql/introduction) · [Window functions](https://duckdb.org/docs/stable/sql/functions/window_functions)
- PostgreSQL: [SELECT/WITH](https://www.postgresql.org/docs/current/queries-with.html) · [Window functions](https://www.postgresql.org/docs/current/functions-window.html)
- W3Schools: [Subquery](https://www.w3schools.com/sql/sql_subqueries.asp) · [CASE](https://www.w3schools.com/sql/sql_case.asp) · [CTE](https://www.w3schools.com/sql/sql_cte.asp) · [Window](https://www.w3schools.com/sql/sql_window.asp)

---

In [ ]:
# §0 เตรียม DB — ใช้ healthinfo.db ต่อ (ถ้าไม่มี รัน week05 + week07 notebook ก่อน)
import sqlite3, pathlib, pandas as pd
db = pathlib.Path("healthinfo.db")
assert db.exists(), "healthinfo.db not found — run week05-sql-basics.ipynb then week07-joins.ipynb"
con = sqlite3.connect(db); cur = con.cursor()
tables = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
for t in tables:
    print(f"  {t:14}", cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
print("sqlite:", sqlite3.sqlite_version)


## §1 Why Insights — คำถามธุรกิจ/คลินิก → query (สไลด์ 02)

ผู้บริหารไม่ได้ถามว่า "SELECT คืออะไร" แต่ถามว่า *"หญิง HbA1c>9 กี่คน ได้ยาไหม? Top-3 รายจ่ายต่อเพศคือใคร? ใครยังไม่เคยได้ยา?"*
ทุกคำถาม = ประกอบร่าง wk05–07 + เครื่องมือใหม่บทนี้

**สูตรคิด 6 ขั้น:** 1) output อะไร → 2) FROM/JOIN → 3) WHERE → 4) subquery/CASE/GROUP → 5) window → 6) ORDER/LIMIT + นับ COUNT เช็ค

In [ ]:
# §1 ตั้ง baseline ก่อนเสมอ: นับแถวทุกตาราง
for t in ["patients", "prescriptions", "referrals"]:
    print(f"{t:14}", cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
# diagnoses มีเฉพาะสัปดาห์ 8 (รัน build/post-08 แล้ว)
print("tables:", [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

## §2 Subquery — query ซ้อน query (สไลด์ 03)

- **Scalar** = ค่าเดียว ใช้เทียบ (`>`, `=`)
- **List** = หลายค่า ใช้กับ `IN`
- **Table** = ใช้ใน `FROM` (ต้องมี alias เสมอ)

In [ ]:
# §2.1 Scalar: ผู้ป่วยที่ HbA1c สูงกว่าค่าเฉลี่ยทั้ง รพ.
display(pd.read_sql("""
SELECT hn, hba1c
FROM patients
WHERE hba1c > (SELECT AVG(hba1c) FROM patients)
ORDER BY hba1c DESC LIMIT 5""", con))

# §2.2 List (IN): คนที่มียา Metformin
display(pd.read_sql("""
SELECT hn FROM patients
WHERE patient_id IN (SELECT patient_id FROM prescriptions WHERE drug='Metformin')
ORDER BY hn LIMIT 5""", con))

# §2.3 Table subquery (FROM): ยาไหนมี avg cost > 50
display(pd.read_sql("""
SELECT drug, avg_cost FROM (
  SELECT drug, AVG(cost_thb) AS avg_cost FROM prescriptions GROUP BY drug
) WHERE avg_cost > 50 ORDER BY avg_cost DESC""", con))

## §3 NOT IN กับดัก NULL vs EXISTS / Anti-JOIN (สไลด์ 04)

หา "ไม่มี/ไม่เคย" — ถ้าลิสต์ใน `NOT IN` มี `NULL` ผลจะได้ **0 แถว** ปลอดภัยกว่าด้วย `NOT EXISTS` หรือ `LEFT JOIN … IS NULL`

In [ ]:
# §3.1 NOT IN (ระวัง NULL) — ได้ 0 แถวถ้าซับคิวรีมี NULL
q_notin = """
SELECT COUNT(*) FROM patients
WHERE patient_id NOT IN (SELECT patient_id FROM prescriptions)"""
print("NOT IN  →", cur.execute(q_notin).fetchone()[0], "rows")

# §3.2 NOT EXISTS (ปลอดภัย)
q_exists = """
SELECT COUNT(*) FROM patients p
WHERE NOT EXISTS (SELECT 1 FROM prescriptions r WHERE r.patient_id = p.patient_id)"""
print("NOT EXISTS →", cur.execute(q_exists).fetchone()[0], "rows")

# §3.3 Anti-join (LEFT … IS NULL) — ต้องเท่ากับ NOT EXISTS
q_anti = """
SELECT COUNT(*) FROM patients p
LEFT JOIN prescriptions r ON r.patient_id = p.patient_id
WHERE r.rx_id IS NULL"""
print("anti-join  →", cur.execute(q_anti).fetchone()[0], "rows")

## §4 Correlated Subquery vs JOIN/CTE (สไลด์ 05)

Correlated = subquery อ้างตารางนอก (รันต่อแถว) — เขียนใหม่เป็น CTE + JOIN ได้เร็วและอ่านง่ายกว่า

In [ ]:
# §4.1 Correlated: ยาที่ราคาแพงกว่าค่าเฉลี่ยของ drug ตัวเอง
display(pd.read_sql("""
SELECT p.hn, r.drug, r.cost_thb
FROM prescriptions r JOIN patients p ON p.patient_id = r.patient_id
WHERE r.cost_thb > (
  SELECT AVG(cost_thb) FROM prescriptions WHERE drug = r.drug  -- ← อ้าง r
) ORDER BY r.cost_thb DESC""", con))

# §4.2 Rewrite ด้วย CTE + JOIN (ผลเหมือนกัน)
display(pd.read_sql("""
WITH avg_by_drug AS (
  SELECT drug, AVG(cost_thb) AS avg_c FROM prescriptions GROUP BY drug
)
SELECT p.hn, r.drug, r.cost_thb
FROM prescriptions r
JOIN avg_by_drug a ON a.drug = r.drug
JOIN patients p ON p.patient_id = r.patient_id
WHERE r.cost_thb > a.avg_c
ORDER BY r.cost_thb DESC""", con))

## §5 CASE WHEN — Bucket ค่าทางคลินิก (สไลด์ 06)

`CASE` = IF ใน SQL — ทำหมวด HbA1c/BP แล้ว `GROUP BY` นับต่อ bucket ได้ทันที (เทียบ `pd.cut`)

In [ ]:
# §5.1 Bucket HbA1c รายคน (ครอบ NULL ก่อนเสมอ)
display(pd.read_sql("""
SELECT hn, hba1c,
  CASE
    WHEN hba1c IS NULL THEN 'ยังไม่ได้วัด'
    WHEN hba1c < 6.5   THEN 'ปกติ'
    WHEN hba1c < 7.0   THEN 'เฝ้าระวัง'
    WHEN hba1c < 9.0   THEN 'คุมไม่ได้'
    ELSE 'วิกฤต >=9'
  END AS a1c_bucket
FROM patients LIMIT 8""", con))

# §5.2 นับต่อ bucket (cross-tab) + sanity check รวม = 100
display(pd.read_sql("""
SELECT CASE
    WHEN hba1c IS NULL THEN 'ยังไม่ได้วัด'
    WHEN hba1c < 6.5   THEN 'ปกติ'
    WHEN hba1c < 7.0   THEN 'เฝ้าระวัง'
    WHEN hba1c < 9.0   THEN 'คุมไม่ได้'
    ELSE 'วิกฤต >=9'
  END AS a1c_bucket,
  COUNT(*) AS n
FROM patients GROUP BY a1c_bucket ORDER BY n DESC""", con))

## §6 CTE (WITH) — ตั้งชื่อ subquery ให้อ่านรู้เรื่อง (สไลด์ 07)

Subquery ซ้อนหลายชั้นอ่านยาก → ตั้งชื่อบนสุด (`clean`, `spend`) แล้ว `SELECT` หลักอ่านสั้น + reuse ได้

In [ ]:
# §6 clean (กรองข้อมูลเพี้ยน) + spend (สรุปรายจ่าย) แล้ว LEFT JOIN
display(pd.read_sql("""
WITH clean AS (
  SELECT patient_id, hn, gender, hba1c
  FROM patients
  WHERE hba1c IS NOT NULL AND birth_date LIKE '____-__-__'
), spend AS (
  SELECT patient_id, SUM(cost_thb) AS total, COUNT(*) AS n_rx
  FROM prescriptions GROUP BY patient_id
)
SELECT c.hn, c.hba1c, s.total
FROM clean c
LEFT JOIN spend s ON s.patient_id = c.patient_id
ORDER BY s.total DESC NULLS LAST LIMIT 5""", con))

## §7 ROW_NUMBER / RANK / DENSE_RANK — Top-N ต่อกลุ่ม (สไลด์ 08)

- `ROW_NUMBER` ไม่ซ้ำ · `RANK` ซ้ำแล้วข้าม · `DENSE_RANK` ซ้ำไม่ข้าม
- **กับดัก:** กรอง window function ใน `WHERE` ของ query เดียวกันไม่ได้ → ห่อ subquery/CTE

In [ ]:
# §7 Top-3 รายจ่ายสูงสุด "ต่อเพศ" (ROW_NUMBER + PARTITION BY)
top3 = pd.read_sql("""
WITH spend AS (
  SELECT patient_id, SUM(cost_thb) AS total FROM prescriptions GROUP BY patient_id
), ranked AS (
  SELECT p.gender, p.hn, s.total,
         ROW_NUMBER() OVER(PARTITION BY p.gender ORDER BY s.total DESC) AS rn,
         RANK()       OVER(PARTITION BY p.gender ORDER BY s.total DESC) AS rk
  FROM spend s JOIN patients p ON p.patient_id = s.patient_id
)
SELECT * FROM ranked WHERE rn <= 3 ORDER BY gender, rn""", con)
display(top3)
print("rows:", len(top3), "(ควรได้ <=6 = 3 ต่อเพศ)")

## §8 Window Aggregate — AVG OVER() (สไลด์ 09)

อยากได้ **รายแถว + ค่าเฉลี่ยกลุ่มในแถวเดียว** — `GROUP BY` ทำไม่ได้ (ยุบแถว) ต้องใช้ `OVER(PARTITION BY …)`

In [ ]:
# §8 เทียบรายจ่ายต่อคน vs ค่าเฉลี่ยเพศเดียวกัน
display(pd.read_sql("""
WITH spend AS (
  SELECT patient_id, SUM(cost_thb) AS total FROM prescriptions GROUP BY patient_id
)
SELECT p.hn, p.gender, s.total,
       AVG(s.total) OVER(PARTITION BY p.gender) AS avg_by_gender,
       s.total - AVG(s.total) OVER(PARTITION BY p.gender) AS diff,
       AVG(s.total) OVER() AS avg_all
FROM spend s JOIN patients p ON p.patient_id = s.patient_id
ORDER BY diff DESC LIMIT 5""", con))

## §9 Cohort Insights — รวมทุกเครื่องมือ (สไลด์ 10)

ตัวอย่าง: **"หญิงอายุ ≥60, HbA1c ≥7, BP ≥140, และไม่เคยได้ Metformin"** — query เดียวตอบได้ (CTE + NOT EXISTS + CASE)

In [ ]:
# §9 cohort ผู้ป่วยหญิงเสี่ยงสูงที่ "ยังไม่ได้ Metformin"
cohort = pd.read_sql("""
WITH base AS (
  SELECT patient_id, hn, gender,
         CAST((julianday('now') - julianday(birth_date)) / 365.25 AS INT) AS age,
         hba1c, systolic_bp
  FROM patients
  WHERE gender='หญิง' AND hba1c >= 7 AND systolic_bp >= 140
    AND birth_date LIKE '____-__-__'
)
SELECT b.hn, b.age, b.hba1c, b.systolic_bp
FROM base b
WHERE NOT EXISTS (
  SELECT 1 FROM prescriptions r WHERE r.patient_id = b.patient_id AND r.drug='Metformin'
)
ORDER BY b.hba1c DESC LIMIT 10""", con)
display(cohort)
print("cohort size:", len(cohort))

## §10 Time Trends — เดือน/ปี (สไลด์ 11)

SQLite ไม่มีชนิด DATE — เก็บเป็น TEXT ISO8601 แล้วใช้ `strftime('%Y-%m', …)` ตัดเดือน
> PostgreSQL: `to_char(rx_date,'YYYY-MM')` · DuckDB: `strftime(rx_date::DATE,'%Y-%m')` (ดู §11)

In [ ]:
# §10 รายจ่ายยาต่อเดือน ( prescriptions มี rx_date จริง )
display(pd.read_sql("""
SELECT strftime('%Y-%m', rx_date) AS ym,
       drug, SUM(cost_thb) AS spend
FROM prescriptions
WHERE rx_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY ym, drug
ORDER BY ym, spend DESC""", con))

## §11 เอนจินเดียว query เดียว: SQLite · DuckDB · PostgreSQL (สไลด์ 12)

`healthinfo.db` (SQLite), `healthinfo.duckdb` (DuckDB) และฐานข้อมูล PostgreSQL `healthinfo`
เก็บข้อมูลชุดเดียวกัน — query มาตรฐาน (CTE + JOIN + window) รันได้เหมือนกัน
สร้างไฟล์/ฐานข้อมูลได้จาก `scripts/` (ดู `scripts/README.md`)

> PostgreSQL ต้องเริ่ม server + รัน `scripts/setup_postgres.sh` ก่อน; ถ้าไม่พร้อม เซลล์จะข้ามอย่างนุ่มนวล

In [ ]:
# §11.1 query มาตรฐาน (portable) — Top-5 รายจ่ายต่อคน
QUERY = """
WITH spend AS (
  SELECT patient_id, SUM(cost_thb) AS total FROM prescriptions GROUP BY patient_id
)
SELECT p.hn, p.gender, s.total
FROM spend s JOIN patients p ON p.patient_id = s.patient_id
ORDER BY s.total DESC
LIMIT 5
"""

# 1) SQLite (ค่าเริ่มต้น)
sqlite_df = pd.read_sql(QUERY, con)
print("SQLite")
display(sqlite_df)

# 2) DuckDB (คอลัมน์/analytics) — ติดตั้งด้วย: uv add duckdb
try:
    import duckdb
    duck_df = duckdb.connect("healthinfo.duckdb").execute(QUERY).df()
    print("DuckDB")
    display(duck_df)
    assert sqlite_df.equals(duck_df), "SQLite กับ DuckDB ให้ผลไม่ตรงกัน!"
    print("SQLite == DuckDB OK")
except ImportError:
    print("duckdb not installed — run: uv add duckdb")

# 3) PostgreSQL (client/server) — ติดตั้งด้วย: uv add 'psycopg[binary]'
try:
    import os, psycopg
    with psycopg.connect(os.environ.get("PGDSN", "dbname=healthinfo")) as conn:
        c = conn.execute(QUERY)
        pg_df = pd.DataFrame(c.fetchall(), columns=[d.name for d in c.description])
    print("PostgreSQL")
    display(pg_df)
    assert sqlite_df.equals(pg_df), "SQLite กับ PostgreSQL ให้ผลไม่ตรงกัน!"
    print("SQLite == PostgreSQL OK")
except Exception as e:
    print("PostgreSQL skipped:", type(e).__name__, "—", e)

In [ ]:
# §11.2 ความต่างเชิงไวยากรณ์ (จดจำให้ขึ้นใจ)
import pandas as pd
diff = pd.DataFrame([
    ("ค่า PK อัตโนมัติ", "INTEGER PRIMARY KEY", "INTEGER PRIMARY KEY", "GENERATED ... AS IDENTITY"),
    ("ทศนิยม", "REAL", "DOUBLE", "DOUBLE PRECISION"),
    ("ตัดเดือนจากวันที่", "strftime('%Y-%m', d)", "strftime(d::DATE,'%Y-%m')", "to_char(d,'YYYY-MM')"),
    ("บังคับ FK", "PRAGMA foreign_keys=ON", "enforced เสมอ", "enforced เสมอ (หรือ NOT VALID)"),
    ("สิทธิ์ระดับผู้ใช้", "ไฟล์ OS", "ไฟล์ OS", "GRANT / ROLE"),
], columns=["เรื่อง", "SQLite", "DuckDB", "PostgreSQL"])
display(diff)

## §12 Lab สัปดาห์ที่ 8 🧪 + Homework

ฝึกบน `healthinfo.db` — ส่งภารกิจในระบบ LMS ตามประกาศ

| # | ภารกิจ | สไลด์ |
|---|---|---|
| 1 | Subquery IN: คนได้ยา Metformin + นับ COUNT เช็ค | 03 |
| 2 | NOT EXISTS / anti-join: คนไม่เคยได้ยาเลย (เทียบ NOT IN trap) | 04 |
| 3 | CASE WHEN: bucket HbA1c → GROUP นับต่อ bucket | 06 |
| 4 | CTE: clean + spend แล้ว LEFT JOIN + ORDER BY total | 07 |
| 5 | Window: Top-3 รายจ่ายสูงสุด *ต่อเพศ* (ROW_NUMBER PARTITION) | 08 |
| 6 | Cohort: หญิง ≥60 + HbA1c≥7 + BP≥140 + NOT Metformin | 10 |
| 7 | รันข้อ 4 บน DuckDB หรือ PostgreSQL แล้วเทียบผล | 12 |

**Self-check**
- [ ] IN vs EXISTS ให้ count เท่ากัน
- [ ] CASE bucket รวม COUNT = 100 คนครบ
- [ ] Top-3 ต่อเพศได้ ≤6 แถว (rn 1–3 ต่อกลุ่ม)
- [ ] query เดียวกันบน SQLite/DuckDB (และ PostgreSQL ถ้าพร้อม) ให้ผลตรงกัน

**Homework 8:** สร้าง insight ใหม่ 1 ข้อ (เช่น "ชายวิกฤต≥9 ที่จ่ายแพงกว่าค่าเฉลี่ยเพศ") ด้วย CTE + Window + CASE พร้อมอธิบาย 3 บรรทัด + แนบ `COUNT()` — และลองรันบนอีกเอนจินหนึ่ง